# 17 · Vision-Language Models

Natural language interaction with satellite imagery: CLIP, Moondream, GeoChat.

## Contents
1. VLMs in the model zoo
2. CLIP zero-shot classification
3. Moondream captioning
4. Visual question answering
5. Sliding window captioning
6. GeoChat conversational AI

## 1 · VLMs in the Model Zoo

In [ ]:
from pygeovision.ai.models.zoo import model_zoo
vlms = model_zoo.filter(task='vlm')
print(f'Vision-Language Models ({len(vlms)}):')
model_zoo.print_table(vlms)

Vision-Language Models (7):

Name                         Task  Architecture      Params(M)  HF
--------------------------------------------------------------------
clip_vit_b32                 vlm   CLIP-ViT-B/32        151.0   ✓
clip_vit_l14                 vlm   CLIP-ViT-L/14        427.0   ✓
openclip_b32                 vlm   OpenCLIP-B/32        151.0   ✓
remoteclip                   vlm   RemoteCLIP           427.0   ✓
geochat                      vlm   GeoChat             7000.0   ✓
moondream2                   vlm   Moondream2          1870.0   ✓
rs5m_clip                    vlm   RS5M-CLIP            427.0   

7 models


## 2 · CLIP Zero-Shot Classification

In [ ]:
PROMPTS = {
    'Urban':       'satellite image of dense urban area with buildings',
    'Forest':      'satellite image of dense forest with tree canopy',
    'Agriculture': 'satellite image of agricultural fields and farmland',
    'Water':       'satellite image of water body lake or river',
    'Barren':      'satellite image of bare soil desert or exposed rock',
    'Grassland':   'satellite image of open grassland or prairie',
}
print('CLIP zero-shot land cover classes:')
for label, prompt in PROMPTS.items():
    print(f'  {label:<14}: "{prompt[:60]}..."')
print()
print('Usage:')
print("  from transformers import CLIPModel, CLIPProcessor")
print("  model = CLIPModel.from_pretrained('openai/clip-vit-large-patch14')")
print("  # Encode image + text -> cosine similarity -> argmax = predicted class")
print()
print('Or via GeoAI:')
print("  labels = client.geoai.classify.clip_classify(img_path, prompts=list(PROMPTS.values()))")
print("  print(f'Predicted: {list(PROMPTS.keys())[labels.dominant_class_idx]}')")
print()
print('Example classification output (forest patch):')
print('  Forest:      0.487 <- predicted')
print('  Grassland:   0.178')
print('  Agriculture: 0.121')
print('  Urban:       0.089')

## 3 · Moondream Image Captioning

In [ ]:
import pygeovision as pgv
client = pgv.PyGeoVision()

print('Moondream satellite captioning examples:')
scenes = [
    ('Urban high-rise district',
     'Dense urban area with grid street pattern. Multiple high-rise buildings '
     'visible. Commercial and residential mixed-use development.'),
    ('Agricultural area',
     'Large rectangular agricultural fields in various cultivation states. '
     'Mechanised farming with visible crop rows and irrigation patterns.'),
    ('Coastal wetland',
     'Coastal wetland with intricate water channels and sediment patterns. '
     'Tidal flats and salt marshes create a mosaic of blue-green hues.'),
]
for scene, caption in scenes:
    print(f'\n  [{scene}]')
    print(f'  Caption: "{caption[:100]}..."')
print()
print('Usage via GeoAI:')
print("  caption = client.geoai.caption.caption('./data/aerial.tif')")
print("  answer  = client.geoai.caption.query('./data/aerial.tif', 'How many buildings?')")
print()
print('Sliding window captioning for large GeoTIFFs:')
print("  results = client.geoai.caption.moondream_sliding_window(")
print("      './data/large.tif', window_size=512, stride=256,")
print("      prompt='What land cover is present?',")
print("      output_path='./results/captions.geojson')")
print("  # Each caption geo-referenced as GeoJSON Feature")

## 4 · Visual Question Answering

In [ ]:
VQA_EXAMPLES = [
    ('How many buildings are visible?',          'Approximately 847 buildings are visible.'),
    ('What is the dominant land use type?',       'Commercial and residential urban development.'),
    ('Are there water bodies visible?',           'A small river is visible in the lower-left.'),
    ('Estimate the green space percentage.',      'Green space covers approximately 12% of the area.'),
    ('What season does this image appear to be?', 'Summer, based on full tree canopy and dry fields.'),
]
print('VQA Examples (Moondream on aerial imagery):')
print()
for q, a in VQA_EXAMPLES:
    print(f'  Q: {q}')
    print(f'  A: {a}')
    print()

## 5 · GeoChat Conversational AI

In [ ]:
from pygeovision.ai.models.zoo import model_zoo
geochat = model_zoo['geochat']
print(f'GeoChat: {geochat.description}')
print(f'Params:  {geochat.params_m}M')
print(f'HF:      {geochat.hf_model_id}')
print()
print('Example conversation:')
dialogue = [
    ('User',   'What do you see in this satellite image?'),
    ('GeoChat','I can see a dense urban environment with high-rise buildings,'
              ' major road networks, and several parks. The city appears to be '
              ' well-developed with mixed commercial and residential zones.'),
    ('User',   'Are there any signs of construction activity?'),
    ('GeoChat','Yes, I can identify three construction sites in the northeast '
              'quadrant, characterised by exposed earth, scaffolding, and '
              'temporary structures. The largest site covers approximately 2 ha.'),
    ('User',   'Estimate the total rooftop solar panel coverage.'),
    ('GeoChat','Based on the reflective surfaces visible on rooftops, I estimate '
              'approximately 3.2% of rooftops have solar panel installations.'),
]
for role, text in dialogue:
    print(f'  {role}: {text[:90]}')